In [ ]:
import os
import re
import math
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment

# ============================================================
# INDEPENDENT IMAGE EVALUATION - SEGMENTATION ONLY
# - Each image is evaluated independently
# - Does NOT use sequences or HOLD/FIRE
# - Does NOT use classification
# - Matches by filename with the GT mask
# - Exports an Excel file with per-image metrics and global metrics
# ============================================================

# ======================= EDIT ================================
os.chdir('/mmsegmentation')

CONFIG = '/mmsegmentation/zmax_configs/for_test_hasta_26_3/bisenet_seg_only_test_clean.py'
CKPT_OR_WORKDIR = '/mmsegmentation/work_dirs/bisenet_seg_only_strict/best_seg_mIoU_iter_8200.pth'
DEVICE = 'cuda:0'


IMG_DIR     = '/automine1d_cls/img_dir/val_aug/'
MASK_DIR    = '/automine1d_cls/ann_dir/val_aug/'
OUTPUT_XLSX = '/mmsegmentation/output/eval_bisenet_seg_only_val.xlsx'

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp'}
MASK_EXT = '.png'
NUM_SEG_CLASSES = 2
IGNORE_INDEX = 255
GT_MASK_BINARY = True
SEG_CLASS_NAMES = ['background', 'road']

PREPROCESS_USE_PINNED = True
PRED_MASK_DTYPE = np.uint8
USE_AUTOCAST = True
# ============================================================

from mmseg.apis import init_model
from mmseg.utils import register_all_modules

try:
    from mmseg.structures import SegDataSample as _TestDataSample
except Exception:
    from mmseg.structures import SegDataSample as _TestDataSample


def ensure_dir_for_file(path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)


def parse_timestamp_from_name(path: str) -> Optional[float]:
    """Extracts a Unix timestamp if it exists in the filename.
    It is not required for evaluation, but it is stored as metadata.
    Supported examples:
      - 1661922943.png
      - 1661922943.00000.png
      - d46844c5-1661922943_clahe.png
    """
    stem = Path(path).stem
    try:
        return float(stem)
    except Exception:
        pass

    matches = re.findall(r'(?<!\d)(\d{10}(?:\.\d+)?)(?!\d)', stem)
    if matches:
        try:
            return float(matches[-1])
        except Exception:
            return None
    return None


def sorted_image_paths(img_dir: str) -> List[str]:
    paths = [str(p) for p in Path(img_dir).iterdir() if p.suffix.lower() in IMG_EXTS]
    return sorted(paths, key=lambda p: Path(p).name)


def resolve_checkpoint(ckpt_or_workdir: str) -> str:
    p = Path(ckpt_or_workdir)
    if p.is_file():
        return str(p)

    if not p.exists():
        raise FileNotFoundError(f'No existe checkpoint ni work_dir: {ckpt_or_workdir}')

    bests = sorted(p.glob('best_seg_mIoU*.pth'))
    if bests:
        return str(bests[-1])

    latest = p / 'latest.pth'
    if latest.exists():
        return str(latest)

    all_pths = sorted(p.glob('*.pth'))
    if all_pths:
        return str(all_pths[-1])

    raise FileNotFoundError(f'No se encontraron .pth en {ckpt_or_workdir}')


def load_gt_mask(mask_path: str) -> np.ndarray:
    gt = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
    if gt is None:
        raise FileNotFoundError(f'No se pudo leer la mascara: {mask_path}')

    if gt.ndim == 3:
        gt = gt[:, :, 0]

    if GT_MASK_BINARY:
        gt = (gt > 0).astype(np.uint8)

    return gt.astype(np.uint8)


def find_mask_path(mask_dir: str, img_name: str) -> str:
    cand1 = os.path.join(mask_dir, img_name)
    if os.path.exists(cand1):
        return cand1
    stem = Path(img_name).stem
    cand2 = os.path.join(mask_dir, stem + MASK_EXT)
    if os.path.exists(cand2):
        return cand2
    raise FileNotFoundError(f'No se encontro mascara para {img_name}')


def fast_hist(gt: np.ndarray, pred: np.ndarray, num_classes: int, ignore_index: int = 255) -> np.ndarray:
    valid = (gt != ignore_index)
    gt = gt[valid].astype(np.int64)
    pred = pred[valid].astype(np.int64)

    valid2 = (gt >= 0) & (gt < num_classes) & (pred >= 0) & (pred < num_classes)
    gt = gt[valid2]
    pred = pred[valid2]

    if gt.size == 0:
        return np.zeros((num_classes, num_classes), dtype=np.int64)

    hist = np.bincount(num_classes * gt + pred, minlength=num_classes ** 2)
    return hist.reshape(num_classes, num_classes)


def ious_from_hist(hist: np.ndarray) -> np.ndarray:
    tp = np.diag(hist).astype(np.float64)
    fp = hist.sum(axis=0) - tp
    fn = hist.sum(axis=1) - tp
    denom = tp + fp + fn
    iou = np.full(tp.shape, np.nan, dtype=np.float64)
    valid = denom > 0
    iou[valid] = tp[valid] / denom[valid]
    return iou


def safe_nanmean(x: np.ndarray) -> float:
    if np.all(np.isnan(x)):
        return float('nan')
    return float(np.nanmean(x))


class NDArraySegInferencer:
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.device = next(self.model.parameters()).device

        cfg_dp = model.cfg.model.get('data_preprocessor', {})
        size = cfg_dp.get('size', (512, 512))
        self.input_w = int(size[0])
        self.input_h = int(size[1])
        self.bgr_to_rgb = bool(cfg_dp.get('bgr_to_rgb', True))
        mean = np.array(cfg_dp.get('mean', [123.675, 116.28, 103.53]), dtype=np.float32)
        std = np.array(cfg_dp.get('std', [58.395, 57.12, 57.375]), dtype=np.float32)

        self.mean = torch.tensor(mean, device=self.device, dtype=torch.float32).view(1, 3, 1, 1)
        self.std = torch.tensor(std, device=self.device, dtype=torch.float32).view(1, 3, 1, 1)

        self._meta = dict(
            ori_shape=(self.input_h, self.input_w),
            img_shape=(self.input_h, self.input_w),
            pad_shape=(self.input_h, self.input_w),
            batch_input_shape=(self.input_h, self.input_w),
            scale_factor=(1.0, 1.0),
            padding_size=[0, 0, 0, 0],
            flip=False,
            flip_direction=None,
        )

        self.use_cuda = (self.device.type == 'cuda')

    def infer(self, img_bgr: np.ndarray) -> np.ndarray:
        if img_bgr is None:
            raise ValueError('img_bgr es None')

        img = cv2.resize(img_bgr, (self.input_w, self.input_h), interpolation=cv2.INTER_LINEAR)
        if self.bgr_to_rgb:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        x = torch.from_numpy(img.transpose(2, 0, 1)).float().unsqueeze(0)
        if PREPROCESS_USE_PINNED and self.use_cuda:
            x = x.pin_memory()
        x = x.to(self.device, non_blocking=self.use_cuda)
        x = (x - self.mean) / self.std

        ds = _TestDataSample()
        ds.set_metainfo(self._meta)

        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=self.use_cuda and USE_AUTOCAST):
                out = self.model.test_step({'inputs': [x[0]], 'data_samples': [ds]})

        if isinstance(out, (list, tuple)):
            sample = out[0]
        else:
            sample = out

        if not hasattr(sample, 'pred_sem_seg') or sample.pred_sem_seg is None:
            raise RuntimeError('No se encontro pred_sem_seg en el DataSample de salida.')

        pred = sample.pred_sem_seg.data
        if pred.ndim == 3 and pred.shape[0] == 1:
            pred = pred[0]
        pred = pred.to(dtype=getattr(torch, str(np.dtype(PRED_MASK_DTYPE).name)))
        pred_mask = pred.detach().cpu().numpy()
        return pred_mask


def prepare_model():
    register_all_modules(init_default_scope=False)
    resolved_ckpt = resolve_checkpoint(CKPT_OR_WORKDIR)
    model = init_model(CONFIG, resolved_ckpt, device=DEVICE)
    model.eval()
    infer = NDArraySegInferencer(model)
    return model, infer, resolved_ckpt


def evaluate_images():
    img_paths = sorted_image_paths(IMG_DIR)
    if len(img_paths) == 0:
        raise RuntimeError(f'No se encontraron imagenes en {IMG_DIR}')

    _, infer, resolved_ckpt = prepare_model()

    results = []
    seg_hist_total = np.zeros((NUM_SEG_CLASSES, NUM_SEG_CLASSES), dtype=np.int64)
    missing_mask = 0

    for idx, img_path in enumerate(tqdm(img_paths, desc='Evaluando imagenes independientes (seg)')):
        img_name = os.path.basename(img_path)
        img_stem = Path(img_name).stem
        timestamp = parse_timestamp_from_name(img_path)

        img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise RuntimeError(f'No se pudo leer la imagen: {img_path}')

        pred_mask = infer.infer(img_bgr)

        mask_path = None
        gt_mask = None
        evaluated = False
        frame_iou = np.full((NUM_SEG_CLASSES,), np.nan, dtype=np.float64)
        frame_miou = float('nan')

        try:
            mask_path = find_mask_path(MASK_DIR, img_name)
            gt_mask = load_gt_mask(mask_path)
            evaluated = True
        except FileNotFoundError:
            missing_mask += 1

        pred_h, pred_w = pred_mask.shape[:2]
        gt_h = gt_w = None

        if evaluated:
            gt_h, gt_w = gt_mask.shape[:2]
            if pred_mask.shape != gt_mask.shape:
                pred_mask_eval = cv2.resize(pred_mask, (gt_w, gt_h), interpolation=cv2.INTER_NEAREST)
            else:
                pred_mask_eval = pred_mask

            hist = fast_hist(gt_mask, pred_mask_eval, NUM_SEG_CLASSES, ignore_index=IGNORE_INDEX)
            seg_hist_total += hist
            frame_iou = ious_from_hist(hist)
            frame_miou = safe_nanmean(frame_iou)
        else:
            pred_mask_eval = pred_mask

        road_ratio_pred = float((pred_mask_eval == 1).mean()) if pred_mask_eval.size > 0 else float('nan')

        results.append(dict(
            idx=idx,
            img_name=img_name,
            img_stem=img_stem,
            timestamp_s=timestamp,
            evaluated=evaluated,
            mask_path=mask_path,
            pred_h=pred_h,
            pred_w=pred_w,
            gt_h=gt_h,
            gt_w=gt_w,
            pred_road_ratio=road_ratio_pred,
            frame_iou_background=float(frame_iou[0]) if NUM_SEG_CLASSES > 0 else float('nan'),
            frame_iou_road=float(frame_iou[1]) if NUM_SEG_CLASSES > 1 else float('nan'),
            frame_mIoU=frame_miou,
        ))

    df = pd.DataFrame(results)
    eval_df = df[df['evaluated'] == True].copy()

    global_ious = ious_from_hist(seg_hist_total)
    summary = dict(
        n_all_images=int(len(df)),
        n_evaluated_images=int(len(eval_df)),
        missing_masks=int(missing_mask),
        mean_frame_mIoU_eval_only=safe_nanmean(eval_df['frame_mIoU'].to_numpy(dtype=np.float64)) if len(eval_df) else float('nan'),
        mean_frame_IoU_background_eval_only=safe_nanmean(eval_df['frame_iou_background'].to_numpy(dtype=np.float64)) if len(eval_df) else float('nan'),
        mean_frame_IoU_road_eval_only=safe_nanmean(eval_df['frame_iou_road'].to_numpy(dtype=np.float64)) if len(eval_df) else float('nan'),
        global_mIoU_from_total_hist_eval_only=safe_nanmean(global_ious),
        global_IoU_background_eval_only=float(global_ious[0]) if len(global_ious) > 0 else float('nan'),
        global_IoU_road_eval_only=float(global_ious[1]) if len(global_ious) > 1 else float('nan'),
        config=CONFIG,
        checkpoint=resolved_ckpt,
        img_dir=IMG_DIR,
        mask_dir=MASK_DIR,
        output_xlsx=OUTPUT_XLSX,
        device=DEVICE,
        model_type='seg_only_independent_images',
    )

    return df, eval_df, seg_hist_total, summary


def style_sheet_basic(ws):
    header_fill = PatternFill(fill_type='solid', fgColor='1F4E78')
    header_font = Font(color='FFFFFF', bold=True)
    center = Alignment(horizontal='center', vertical='center')

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = center

    for col_cells in ws.columns:
        max_len = 0
        for cell in col_cells:
            val = '' if cell.value is None else str(cell.value)
            max_len = max(max_len, len(val))
        ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 2, 40)


def write_excel(df: pd.DataFrame, eval_df: pd.DataFrame, seg_hist_total: np.ndarray, summary: Dict[str, object], out_path: str):
    ensure_dir_for_file(out_path)

    hist_df = pd.DataFrame(
        seg_hist_total,
        index=[f'GT_{c}' for c in SEG_CLASS_NAMES],
        columns=[f'Pred_{c}' for c in SEG_CLASS_NAMES],
    )
    summary_df = pd.DataFrame({'metric': list(summary.keys()), 'value': list(summary.values())})

    with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='PerImage', index=False)
        eval_df.to_excel(writer, sheet_name='EvaluatedOnly', index=False)
        summary_df.to_excel(writer, sheet_name='Summary', index=False)
        hist_df.to_excel(writer, sheet_name='TotalHist')

    wb = load_workbook(out_path)
    for sheet_name in ['PerImage', 'EvaluatedOnly', 'Summary', 'TotalHist']:
        ws = wb[sheet_name]
        style_sheet_basic(ws)

    for sheet_name in ['PerImage', 'EvaluatedOnly']:
        ws = wb[sheet_name]
        headers = {cell.value: cell.column for cell in ws[1] if cell.value}
        for col_name, fmt in {
            'timestamp_s': '0.000000',
            'pred_road_ratio': '0.0000',
            'frame_iou_background': '0.0000',
            'frame_iou_road': '0.0000',
            'frame_mIoU': '0.0000',
        }.items():
            if col_name in headers:
                col_idx = headers[col_name]
                for row in range(2, ws.max_row + 1):
                    ws.cell(row=row, column=col_idx).number_format = fmt

    wb.save(out_path)


def main():
    df, eval_df, seg_hist_total, summary = evaluate_images()
    write_excel(df, eval_df, seg_hist_total, summary, OUTPUT_XLSX)

    print('\n=== RESUMEN ===')
    for k, v in summary.items():
        print(f'{k}: {v}')

    print(f'\nExcel guardado en: {OUTPUT_XLSX}')


main()
